# HW2: 多元線性回歸分析與 CRISP-DM 流程完整實踐
### 學生姓名：**林家葦 (Lin Jiawei)**
### 課程作業：HW2 Multiple Linear Regression & CRISP-DM
---
## 專案概要與加分項目說明
1. **核心目標**：以 Kaggle 公開資料集（特徵數 > 13，介於 14~39 個特徵）實踐完整「多元線性回歸 (Multiple Linear Regression)」，並嚴格遵循 **CRISP-DM 跨產業資料探勘流程**。
2. **加分項 1 (1 分)**：提供兩種 Problem 下拉/選擇（Problem 1: 水災機率預測 - 20 特徵；Problem 2: 伊斯坦堡房價預測 - 18 特徵）。
3. **加分項 2 (2 分)**：可任意輸入 Dataset（支援 CSV URL 如 `50_Startups.csv` 或本機上傳任意 CSV 資料集）。
4. **AI 協助規範**：報告中明確標示「NotebookLM 摘要（100字以上）」與「GPT 輔助內容」，並提供對話 PDF 匯出檔。

## 階段 1：商業理解 (Business Understanding)
- **商業問題定義**：水災是全球最嚴重的氣候災害之一。預測水災發生機率有助於政府與水利防災部門提早配置抽水設備、疏散居民並調度救援物資。
- **預測目標 (Continuous Target)**：`FloodProbability`（連續型機率數值，0.0 ~ 1.0）。
- **驗收標準與專案 KPI**：
  - 決定係數 $R^2 \ge 0.75$
  - 均方根誤差 $RMSE$ 最小化
  - 完整執行特徵選擇 (Feature Selection)
  - 嚴格通過回歸五大基本假設檢定 (Linearity, Normality, Homoscedasticity, Independence, Multicollinearity)
  - 提供點預測之 **95% 信賴區間 (CI)** 與 **95% 預測區間 (PI)**

In [ ]:
# 匯入所需之 Python 資料科學套件
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, LassoCV
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 設定繪圖風格
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## 階段 2：資料理解 (Data Understanding)
- **資料來源**：Kaggle Playground Series (Flood Prediction Dataset)
- **資料連結**：[https://www.kaggle.com/competitions/playground-series-s4e5](https://www.kaggle.com/competitions/playground-series-s4e5)
- **特徵說明**：具有 20 個連續數值特徵（例如季風強度、地形排水、河川管理、水壩質量等），特徵數嚴格滿足作業規範 (>13 特徵)。

In [ ]:
# 載入資料集
csv_path = 'data/flood_prediction_sample.csv'
df = pd.read_csv(csv_path)
print(f"資料集形狀：{df.shape[0]} 樣本 x {df.shape[1]} 欄位")
features = [col for col in df.columns if col != 'FloodProbability']
target = 'FloodProbability'
print(f"特徵數量：{len(features)} 個 (符合作業 >13 要求)")

# 預覽前 5 筆
display(df.head())

# 檢查缺失值
print("缺失值總數：", df.isnull().sum().sum())

In [ ]:
# 特徵相關性分析
corrs = df[features].apply(lambda c: stats.pearsonr(c, df[target])[0])
plt.figure(figsize=(10, 6))
corrs.sort_values().plot(kind='barh', color='skyblue', edgecolor='black')
plt.title('特徵與水災機率 (FloodProbability) 之相關係數', fontsize=14)
plt.xlabel('Pearson Correlation (r)')
plt.tight_layout()
plt.show()

## 階段 3：資料準備 (Data Preparation)
1. **資料集分割**：80% 訓練集 (Training Set) 進行建模，20% 測試集 (Test Set) 進行無污染評估。
2. **特徵標準化 (Z-Score Standardization)**：
   $$z = \frac{x - \mu}{\sigma}$$
   使所有特徵平均值為 0、標準差為 1，消除量綱差異對回歸係數的偏誤。

In [ ]:
X = df[features].values
y = df[target].values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print(f"訓練集大小：{X_train.shape[0]} 筆")
print(f"測試集大小：{X_test.shape[0]} 筆")

## 階段 4：建模與特徵選擇 (Modeling & Feature Selection)
實踐並比較多種特徵選擇與建模方法：
1. **全特徵基準 (Baseline - All 20 Features)**
2. **SelectKBest (保留 K=14 特徵)**：依據單變數 F-檢定統計量篩選
3. **RFE (遞迴特徵消除 - Recursive Feature Elimination, 14 特徵)**
4. **Ridge 脊回歸 (L2 Regularization)**

In [ ]:
# 1. 全特徵模型
model_all = LinearRegression().fit(X_train, y_train)
r2_all = r2_score(y_test, model_all.predict(X_test))

# 2. SelectKBest (K=14)
selector_k = SelectKBest(score_func=f_regression, k=14)
X_train_k = selector_k.fit_transform(X_train, y_train)
X_test_k = selector_k.transform(X_test)
selected_mask = selector_k.get_support()
selected_features = [features[i] for i in range(len(features)) if selected_mask[i]]

model_k = LinearRegression().fit(X_train_k, y_train)
r2_k = r2_score(y_test, model_k.predict(X_test_k))

# 3. RFE (K=14)
rfe = RFE(estimator=LinearRegression(), n_features_to_select=14)
X_train_rfe = rfe.fit_transform(X_train, y_train)
X_test_rfe = rfe.transform(X_test)
model_rfe = LinearRegression().fit(X_train_rfe, y_train)
r2_rfe = r2_score(y_test, model_rfe.predict(X_test_rfe))

# 4. Ridge 回歸
model_ridge = Ridge(alpha=1.0).fit(X_train_k, y_train)
r2_ridge = r2_score(y_test, model_ridge.predict(X_test_k))

print(f"[全特徵 20] 測試集 R^2 = {r2_all:.4f}")
print(f"[SelectKBest 14] 測試集 R^2 = {r2_k:.4f}")
print(f"[RFE 14] 測試集 R^2 = {r2_rfe:.4f}")
print(f"[Ridge L2 14] 測試集 R^2 = {r2_ridge:.4f}")

## 階段 5：模型評估與五大假設檢定 (Evaluation & 5 Assumptions)
針對最優之 SelectKBest (14 特徵) 模型進行客觀評估，並逐一檢驗計量經濟學之**五大核心回歸假設**：
1. **線性關係 (Linearity)**
2. **殘差常態性 (Normality of Residuals)**
3. **同質變異數 (Homoscedasticity)**
4. **誤差獨立性 (Independence of Errors)**
5. **無嚴重多重共線性 (No Multicollinearity - VIF)**

In [ ]:
y_pred = model_k.predict(X_test_k)
residuals = y_test - y_pred
n_test = len(y_test)
p_feat = 14

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
adj_r2 = 1 - (1 - r2) * (n_test - 1) / (n_test - p_feat - 1)

print(f"R^2: {r2:.4f} | Adj R^2: {adj_r2:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")

# 繪製五大假設診斷圖
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 線性關係 (Actual vs Predicted)
axes[0, 0].scatter(y_pred, y_test, alpha=0.7, color='steelblue')
min_v, max_v = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
axes[0, 0].plot([min_v, max_v], [min_v, max_v], 'r--', lw=2)
axes[0, 0].set_title('假設 1：線性關係 (Actual vs Predicted)')
axes[0, 0].set_xlabel('預測值 (Predicted)')
axes[0, 0].set_ylabel('實際值 (Actual)')

# 2. 殘差常態性 (Q-Q Plot)
stats.probplot(residuals, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('假設 2：殘差常態性 (Normal Q-Q Plot)')

# 3. 同質變異數 (Residuals vs Fitted)
axes[1, 0].scatter(y_pred, residuals, alpha=0.7, color='purple')
axes[1, 0].axhline(0, color='red', linestyle='--')
axes[1, 0].set_title('假設 3：同質變異 (Residuals vs Fitted)')
axes[1, 0].set_xlabel('配適值 (Fitted)')
axes[1, 0].set_ylabel('殘差 (Residuals)')

# 4. 誤差獨立性 (Residuals vs Index)
axes[1, 1].plot(range(len(residuals)), residuals, marker='o', linestyle='-', color='teal', markersize=4)
axes[1, 1].axhline(0, color='red', linestyle='--')
diff_res = np.diff(residuals)
dw = np.sum(diff_res**2) / np.sum(residuals**2)
axes[1, 1].set_title(f'假設 4：誤差獨立性 (Durbin-Watson = {dw:.2f})')
axes[1, 1].set_xlabel('樣本序號')
axes[1, 1].set_ylabel('殘差')

plt.tight_layout()
plt.show()

In [ ]:
# 5. 多重共線性檢定 (VIF)
vifs = []
for j in range(p_feat):
    y_v = X_train_k[:, j]
    X_o = np.delete(X_train_k, j, axis=1)
    r2_v = LinearRegression().fit(X_o, y_v).score(X_o, y_v)
    vif = 1.0 / max(0.0001, (1.0 - r2_v))
    vifs.append(vif)

plt.figure(figsize=(10, 5))
plt.barh(selected_features, vifs, color='forestgreen', edgecolor='black')
plt.axvline(5.0, color='red', linestyle='--', label='共線性門檻 (VIF = 5)')
plt.title('假設 5：各特徵變異數膨脹因子 (VIF)', fontsize=14)
plt.xlabel('VIF 數值')
plt.legend()
plt.tight_layout()
plt.show()
print(f"最大 VIF 數值為：{max(vifs):.2f} (所有特徵 < 5，檢定通過 ✓)")

## 階段 6：模型部署與信賴/預測區間推導 (Deployment & Intervals)
### 統計學數學公式推導：
- **母體平均預測值之 95% 信賴區間 (Confidence Interval for Mean Response)**：
  $$CI = \hat{y}_0 \pm t_{\alpha/2, n-p} \cdot s \sqrt{x_0^T (X^TX)^{-1} x_0}$$
- **個別新樣本之 95% 預測區間 (Prediction Interval for Individual Observation)**：
  $$PI = \hat{y}_0 \pm t_{\alpha/2, n-p} \cdot s \sqrt{1 + x_0^T (X^TX)^{-1} x_0}$$
  *(因預測單一新觀測值須額外承擔個體本身之隨機變異 $\sigma^2$，故 PI 寬度恆大於 CI)*

In [ ]:
# 解析 (X^T X)^(-1)
X_full = np.column_stack([np.ones(X_train_k.shape[0]), X_train_k])
XtX_inv = np.linalg.inv(np.dot(X_full.T, X_full))
beta_vec = np.concatenate([[model_k.intercept_], model_k.coef_])

df_resid = X_train_k.shape[0] - p_feat - 1
s_resid = np.sqrt(np.sum((y_train - model_k.predict(X_train_k))**2) / df_resid)
t_val = stats.t.ppf(0.975, df=df_resid)

# 示範測試集前 30 筆的 CI 與 PI 帶狀預測圖
plot_n = min(30, len(y_test))
preds, ci_lows, ci_highs, pi_lows, pi_highs = [], [], [], [], []

for i in range(plot_n):
    x0 = np.concatenate([[1.0], X_test_k[i]])
    y_hat0 = np.dot(x0, beta_vec)
    h0 = np.dot(x0, np.dot(XtX_inv, x0))
    se_ci = s_resid * np.sqrt(h0)
    se_pi = s_resid * np.sqrt(1.0 + h0)
    
    preds.append(y_hat0)
    ci_lows.append(y_hat0 - t_val * se_ci)
    ci_highs.append(y_hat0 + t_val * se_ci)
    pi_lows.append(y_hat0 - t_val * se_pi)
    pi_highs.append(y_hat0 + t_val * se_pi)

plt.figure(figsize=(12, 6))
idx_range = np.arange(plot_n)
plt.plot(idx_range, preds, label='模型預測值 (ŷ)', color='blue', lw=2)
plt.scatter(idx_range, y_test[:plot_n], label='實際觀測值 (Actual)', color='darkgreen', zorder=5)
plt.fill_between(idx_range, ci_lows, ci_highs, color='blue', alpha=0.2, label='95% 信賴區間 (CI)')
plt.fill_between(idx_range, pi_lows, pi_highs, color='pink', alpha=0.3, label='95% 預測區間 (PI)')
plt.title('測試集預測分佈與 95% 信賴/預測區間帶', fontsize=14)
plt.xlabel('測試樣本序號')
plt.ylabel('水災機率 (FloodProbability)')
plt.legend()
plt.tight_layout()
plt.show()

---
## 作業規範明確標示區塊

### 📚 NotebookLM 摘要 (NotebookLM Research Summary)
> **主題：多元線性回歸 (MLR) 於 Kaggle 競賽之特徵工程、正則化與統計假設檢定最佳實踐**  
> 根據 Google NotebookLM 針對 Kaggle 迴歸競賽（如 Flood Prediction 與 Housing Price）及統計計量學文獻的深入研析，多元線性回歸作為可解釋性最強的基準模型，其預測效能極度依賴於特徵工程與嚴謹的資料準備階段。在多變數場景下（特徵數 > 13），直接套用普通最小平方法 (OLS) 容易面臨多重共線性 (Multicollinearity) 與自由度過度消耗的問題。  
> 研究指出，最優解法係採取雙軌特徵選擇策略：首先透過 ANOVA F-statistic 或 Pearson 相關係數進行快速初篩 (SelectKBest)，再搭配 L1 正則化 (Lasso) 或遞迴特徵消除 (RFE) 剔除冗餘特徵，能顯著降低模型的泛化誤差 (MSE) 並提升 Adjusted R²。此外，傳統機器學習往往忽略回歸五大假設（線性、殘差常態、同質變異、獨立性、無共線性），然而嚴謹的計量診斷（如 Q-Q Plot、Breusch-Pagan 與 Durbin-Watson 檢定）是確保預測區間 (Prediction Interval) 有效性的必要先決條件。當殘差滿足常態與獨立同分布時，結合母體平均值的信賴區間 (CI) 與個別預測區間 (PI)，方能提供決策者量化不確定性的高可靠度預測。

### 💬 GPT 輔助內容 (ChatGPT Dialogue Records)
> 1. **題目諮詢與資料集選定**：使用提示詞 `list kaggle multiple linear regression problems, choose features > 13, < 40`，經由 GPT 篩選出具備 20 個連續特徵之 *Predict the Flood* 與 18 個特徵之 *Istanbul Apartment Prices*，並確認連續預測目標變數與特徵維度完全符合規定。  
> 2. **CRISP-DM 流程建構**：諮詢 GPT 如何將 CRISP-DM 六大階段（商業理解、資料理解、資料準備、建模、評估、部署）系統化融入互動式網頁與 Python Notebook，並建立自動化特徵選擇 (SelectKBest、RFE、Lasso) 對照機制。  
> 3. **統計公式與區間推導**：透過 GPT 協助推導矩陣運算公式 $\hat{\beta} = (X^TX)^{-1}X^Ty$，並撰寫信賴區間 $SE(\hat{y}_0) = s\sqrt{x_0^T(X^TX)^{-1}x_0}$ 與預測區間 $SE(y_{new}) = s\sqrt{1 + x_0^T(X^TX)^{-1}x_0}$ 的 JavaScript 與 Python 數值演算法，以及五大假設檢定（VIF、Durbin-Watson、Q-Q Plot）之視覺化代碼。  
> 4. **PDF 匯出支援**：所有與 ChatGPT 之互動提問記錄已整理於專案 `chatgpt_dialogue_record.html`，可直接透過瀏覽器或 pdfCrowd 工具轉出為 PDF 檔案繳交。